# CDN4: SISO vs better_path

This notebook compares the full path sets produced by `SISO.jl` and `better_path.jl` on the `CDN4` network, and then checks whether the conditions agree on the overlapping paths.

It also probes a few internal `(from, to)` pair queries to explain why the two full path sets are currently different.

In [ ]:
using BindingAndCatalysis
using Polyhedra
using Logging

function cdn4_model()
    N = [1 1 0 0 -1 0 0 0 0 0;
         1 0 1 0 0 -1 0 0 0 0;
         1 0 0 1 0 0 -1 0 0 0;
         0 1 1 0 0 0 0 -1 0 0;
         0 1 0 1 0 0 0 0 -1 0;
         0 0 1 1 0 0 0 0 0 -1]
    return Bnc(N = N)
end

function collect_siso_paths(siso)
    polys = get_polyhedra(siso)
    out = Dict{Tuple{Vararg{Int}}, Polyhedra.Polyhedron}()
    for (path, poly) in zip(siso.rgm_paths, polys)
        out[Tuple(path)] = poly
    end
    return out
end

function collect_better_paths(rg)
    out = Dict{Tuple{Vararg{Int}}, Polyhedra.Polyhedron}()
    for s in rg.sources, t in rg.sinks
        ps = rg.paths[s, t]
        ps === nothing && continue
        for p in ps
            out[Tuple(p.path)] = p.condition
        end
    end
    return out
end

reduce_better(poly) = eliminate(poly, 1)

function compare_common_conditions(siso_map, better_map)
    common = intersect(Set(keys(siso_map)), Set(keys(better_map)))
    mismatches = Tuple{Vararg{Int}}[]
    for key in sort(collect(common))
        same = same_polyhedron(siso_map[key], reduce_better(better_map[key]))
        if same == false
            mismatches = [mismatches; (key,)]
            if length(mismatches) >= 20
                break
            end
        end
    end
    return length(common), mismatches
end

function singleton_common_vertices(g)
    return sort(collect(intersect(get_sources(g), get_sinks(g))))
end

function force_pair_solver(rg)
    solver_name = Symbol("_better_path_finder" * string(Char(33)))
    return getfield(BindingAndCatalysis, solver_name)
end

function pair_paths(rg, solver, s, t)
    solver(rg, s, t)
    ps = rg.paths[s, t]
    return ps === nothing ? Tuple{Vararg{Int}}[] : Tuple.(getfield.(ps, :path))
end

In [ ]:
model = cdn4_model()

siso, rg, siso_map, better_map = with_logger(NullLogger()) do
    siso_local = SISOPaths(model, 1)
    rg_local = better_path_finder(model, [1.0, 0.0, 0.0, 0.0])
    siso_map_local = collect_siso_paths(siso_local)
    better_map_local = collect_better_paths(rg_local)
    (siso_local, rg_local, siso_map_local, better_map_local)
end

siso_set = Set(keys(siso_map))
better_set = Set(keys(better_map))
common_paths = intersect(siso_set, better_set)
siso_only = sort(collect(setdiff(siso_set, better_set)))
better_only = sort(collect(setdiff(better_set, siso_set)))

println("SISO path count: ", length(siso_set))
println("better_path count: ", length(better_set))
println("Common paths: ", length(common_paths))
println("SISO-only paths: ", length(siso_only))
println("better-only paths: ", length(better_only))
println()
println("First 10 better-only paths:")
println(better_only[1:min(end, 10)])
println()
println("First 10 SISO-only paths:")
println(siso_only[1:min(end, 10)])

The `better-only` paths are expected to be singletons if they come from isolated vertices that are simultaneously sources and sinks in the raw graph.

`SISOPaths(model, change_qK)` filters these singular isolated vertices out via `get_sources_sinks(model, g)`, while `better_path_finder` currently keeps them and therefore returns the trivial path `[v]` for each such vertex.

In [ ]:
common_singletons = singleton_common_vertices(siso.qK_grh)

println("Count of raw graph vertices that are both source and sink: ", length(common_singletons))
println("First 20 such vertices: ", common_singletons[1:min(end, 20)])
println()
println("All better-only paths have length 1: ", all(length(path) == 1 for path in better_only))
println("better-only vertex list matches the raw source∩sink list: ", [only(path) for path in better_only] == common_singletons)

Next, compare the polyhedral conditions for all overlapping paths.

For `better_path`, we eliminate dimension `1` so that its condition lives in the same space as the `SISO` path condition.

In [ ]:
ncommon, mismatches = compare_common_conditions(siso_map, better_map)

println("Number of common paths checked: ", ncommon)
println("Condition mismatches on common paths: ", length(mismatches))
println("First mismatches (if any): ", mismatches)

The remaining gap is the `223` nontrivial paths that `SISO` has but `better_path` does not.

A minimal probe shows the reason: `_better_path_finder!` only returns the direct path `[from, to]` when `pass_by` is empty. If there is at least one intermediate bridge candidate, the current implementation skips the direct edge path entirely, even when the edge `from -> to` exists.

So if a pair like `(23, 22)` has both a direct edge `23 -> 22` and a longer route `23 -> 21 -> 22`, the current solver keeps the longer route but drops the direct one.

In [ ]:
solver = force_pair_solver(rg)

println("Paths for pair (23, 22):")
println(pair_paths(rg, solver, 23, 22))
println()
println("Paths for pair (39, 23):")
println(pair_paths(rg, solver, 39, 23))
println()
println("Paths for pair (39, 22):")
println(pair_paths(rg, solver, 39, 22))

Interpretation:

- `(23, 22)` should include both `(23, 22)` and `(23, 21, 22)`, but the solver only returns the longer one.
- `(39, 23)` should include both `(39, 23)` and `(39, 7, 23)`, but the solver only returns the longer one.
- Consequently, `(39, 22)` misses the path `(39, 23, 22)`, and this omission propagates upward to source-to-sink paths such as `(35, 39, 23, 22)`.

So the current difference in full path sets has two components:

1. `better_path` has `48` extra singleton paths from isolated source∩sink vertices that `SISO` intentionally filters out.
2. `better_path` misses `223` nontrivial paths because the direct edge path is omitted whenever `pass_by` is nonempty.

For the `3713` paths that do overlap, the reduced `better_path` condition and the `SISO` condition agree exactly.